### **Autor:** David Roca Tauste

---
---
# **📌ACTIVIDAD 1: UN CLASIFICADOR NAÏVE BAYES DESDE CERO**
---
---

## DATOS DE TRAIN Y TEST

In [12]:
train_spam = ['send us your password', 'review our website', 'send your password', 'send us your account']
train_ham = ['Your activity report', 'benefits physical activity', 'the importance vows']
test_emails = {'spam': ['renew your password', 'renew your vows'],
               'ham': ['benefits of our account', 'the importance of physical activity']}

In [13]:
# Hacer un vocabulario de palabras únicas que aparecen en los mails spam
vocab_palabras_spam = []
for frase in train_spam:
    frase_como_lista = frase.split()
    for w in frase_como_lista:
        vocab_palabras_spam.append(w)
print(vocab_palabras_spam)

['send', 'us', 'your', 'password', 'review', 'our', 'website', 'send', 'your', 'password', 'send', 'us', 'your', 'account']


In [14]:
vocab_palabras_spam_unicas = list(dict.fromkeys(vocab_palabras_spam))
print(vocab_palabras_spam_unicas)

['send', 'us', 'your', 'password', 'review', 'our', 'website', 'account']


In [15]:
# Probabilidades de cada palabra condicionado a spam
dict_spamnicidad = {}
for w in vocab_palabras_spam_unicas:
    mails_con_w = 0  # contador
    for frase in train_spam:
        if w in frase:
            mails_con_w += 1
    print(f"Número de mails spam con la palabra '{w}': {mails_con_w}")
    spamnicidad = (mails_con_w + 1) / (len(train_spam) + 2)  # suavizado
    print(f"Spamnicidad de la palabra '{w}': {spamnicidad}\n")
    dict_spamnicidad[w.lower()] = spamnicidad

Número de mails spam con la palabra 'send': 3
Spamnicidad de la palabra 'send': 0.6666666666666666

Número de mails spam con la palabra 'us': 2
Spamnicidad de la palabra 'us': 0.5

Número de mails spam con la palabra 'your': 3
Spamnicidad de la palabra 'your': 0.6666666666666666

Número de mails spam con la palabra 'password': 2
Spamnicidad de la palabra 'password': 0.5

Número de mails spam con la palabra 'review': 1
Spamnicidad de la palabra 'review': 0.3333333333333333

Número de mails spam con la palabra 'our': 4
Spamnicidad de la palabra 'our': 0.8333333333333334

Número de mails spam con la palabra 'website': 1
Spamnicidad de la palabra 'website': 0.3333333333333333

Número de mails spam con la palabra 'account': 1
Spamnicidad de la palabra 'account': 0.3333333333333333



## ENTREGA 1: completa el código y calcula la hamicidad de las palabras creando un diccionario Python donde cada clave sea una de las palabras que aparecen en los datos de mail ham. Si lo imprimes debe darte este resultado:
Hamicidad: {'your': 0.4, 'activity': 0.6, 'report': 0.4, 'benefits': 0.4, 'physical': 0.4, 'the': 0.4, 'importance': 0.4, 'vows': 0.4}<br><br><br>
Se encuentra abajo.

In [16]:
# Hacer un vocabulario de palabras únicas que aparecen en los mails ham
vocab_palabras_ham = []
for frase in train_ham:
    frase_como_lista = frase.lower().split()
    for w in frase_como_lista:
        vocab_palabras_ham.append(w)
print(vocab_palabras_spam)

vocab_palabras_ham_unicas = list(dict.fromkeys(vocab_palabras_ham))
print(vocab_palabras_spam_unicas)

# Probabilidades de cada palabra condicionado a ham
dict_hamicidad = {}
for w in vocab_palabras_ham_unicas:
    mails_con_w = 0
    for frase in train_ham:
        if w in frase.lower():
            mails_con_w += 1
    hamicidad = (mails_con_w + 1) / (len(train_ham) + 2)
    dict_hamicidad[w] = round(hamicidad, 1)

['send', 'us', 'your', 'password', 'review', 'our', 'website', 'send', 'your', 'password', 'send', 'us', 'your', 'account']
['send', 'us', 'your', 'password', 'review', 'our', 'website', 'account']


In [17]:
print("Hamicidad:", dict_hamicidad)

Hamicidad: {'your': 0.4, 'activity': 0.6, 'report': 0.4, 'benefits': 0.4, 'physical': 0.4, 'the': 0.4, 'importance': 0.4, 'vows': 0.4}


---

In [18]:
# Probabilidades de spam y ham
prob_spam = len(train_spam) / (len(train_spam) + len(train_ham))
prob_ham = len(train_ham) / (len(train_spam) + len(train_ham))
print(f"P(S) = {prob_spam}")
print(f"P(H) = {prob_ham}")

P(S) = 0.5714285714285714
P(H) = 0.42857142857142855


In [19]:
# Dividir los mail en palabras únicas
distintas_palabras_como_frase_test = []
for frase in test_emails["spam"] + test_emails["ham"]:
    frase_como_lista = frase.split()
    sentencia = []
    for w in frase_como_lista:
        sentencia.append(w)
    distintas_palabras_como_frase_test.append(sentencia)
print(distintas_palabras_como_frase_test)
test_spam_tokenizado = distintas_palabras_como_frase_test[: len(test_emails["spam"])]
test_ham_tokenizado = distintas_palabras_como_frase_test[len(test_emails["spam"]) :]
print("Test spam tokenizado", test_spam_tokenizado)
print("Test ham tokenizado", test_ham_tokenizado)

[['renew', 'your', 'password'], ['renew', 'your', 'vows'], ['benefits', 'of', 'our', 'account'], ['the', 'importance', 'of', 'physical', 'activity']]
Test spam tokenizado [['renew', 'your', 'password'], ['renew', 'your', 'vows']]
Test ham tokenizado [['benefits', 'of', 'our', 'account'], ['the', 'importance', 'of', 'physical', 'activity']]


In [20]:
# Eliminar palabras de test sin datos en los datos de train
spam_test_reducido = []
for frase in test_spam_tokenizado:
    palabras_ = []
    for w in frase:
        if w in vocab_palabras_spam_unicas:
            print(f"'{w}', ok")
            palabras_.append(w)
        elif (
            w in vocab_palabras_ham_unicas
        ):
            print(f"'{w}', ok")
            palabras_.append(w)
        else:
            print(f"'{w}', sin información en train como spam")
    spam_test_reducido.append(palabras_)
print("Test de spam reducido:", spam_test_reducido)

'renew', sin información en train como spam
'your', ok
'password', ok
'renew', sin información en train como spam
'your', ok
'vows', ok
Test de spam reducido: [['your', 'password'], ['your', 'vows']]


## ENTREGA 2: completa el código y haz lo mismo para los datos de test etiquetados como ham, eliminando las palabras que no estén en los datos de train como ham. Si lo imprimes, debe darte este resultado:
Test de ham reducido: [['benefits', 'our', 'account'], ['the', 'importance', 'physical', 'activity']]<br><br>

Se encuentra abajo.

In [21]:
# Eliminar palabras de test sin datos en los datos de train
ham_test_reducido = []
for frase in test_ham_tokenizado:
    palabras_ = []
    for w in frase:
        if w in vocab_palabras_spam_unicas:
            print(f"'{w}', ok")
            palabras_.append(w)
        elif w in vocab_palabras_ham_unicas:
            print(f"'{w}', ok")
            palabras_.append(w)
        else:
            print(f"'{w}', sin información en train como ham")
    ham_test_reducido.append(palabras_)

'benefits', ok
'of', sin información en train como ham
'our', ok
'account', ok
'the', ok
'importance', ok
'of', sin información en train como ham
'physical', ok
'activity', ok


In [22]:
print("Test de ham reducido:", ham_test_reducido)

Test de ham reducido: [['benefits', 'our', 'account'], ['the', 'importance', 'physical', 'activity']]


---
## STEMMING (ELIMINAR ELEMENTOS POCO IMPORTANTES)

In [23]:
# stemmed
test_spam_stemmed = []
poco_importantes = ["us", "the", "of", "your"]  # palabras no clave
for email in spam_test_reducido:
    email_limpiado = []
    for w in email:
        if w in poco_importantes:
            print(f"Eliminar '{w}'")
        else:
            email_limpiado.append(w)
    test_spam_stemmed.append(email_limpiado)
    
print("Test spam stemmed:", test_spam_stemmed)

Eliminar 'your'
Eliminar 'your'
Test spam stemmed: [['password'], ['vows']]


## ENTREGA 3: completa el código y haz lo mismo para los datos de test y consigue test_ham_stemmed. Si lo imprimes, debe darte este resultado:
Eliminar "the"<br>
Test ham stemmed: [['benefits', 'our', 'account'], ['importance', 'physical', 'activity']]<br><br>

Se encuentra abajo.

In [24]:
# stemmed
test_ham_stemmed = []
poco_importantes = ["us", "the", "of", "your"]  # palabras no clave
for email in ham_test_reducido:
    email_limpiado = []
    for w in email:
        if w in poco_importantes:
            print(f"Eliminar '{w}'")
        else:
            email_limpiado.append(w)
    test_ham_stemmed.append(email_limpiado)
    
print("Test spam stemmed:", test_ham_stemmed)

Eliminar 'the'
Test spam stemmed: [['benefits', 'our', 'account'], ['importance', 'physical', 'activity']]


---
## CALCULAR BAYES

In [25]:
def multiplica(lista): # multiplica las probs de las palabras de la lista
    total_prob = 1
    for i in lista:
        total_prob = total_prob * i
    return total_prob

def Bayes(email):
    probs = [] # para cada palabra w del mail
    PS = prob_spam
    print(f'P(S)={PS}')
    try:
        p_ws = dict_spamnicidad[w]
        print(f'P(w|spam)={p_ws}')
    except KeyError:
        p_ws = 1 / (len(train_spam) + 2) # Aplicar suavizado a palabras no vistas en spam
        print(f'P(w|spam)={p_ws}')
    PH = prob_ham
    print(f'P(H)={PH}')
    try:
        p_wh = dict_hamicidad[w]
        print(f'P(w|ham)={p_wh}')
    except KeyError:
        p_wh = 1 / (len(train_ham) + 2) # Aplicar smoothing
        print(f'P(w|ham)={p_wh}')
    p_spam_BAYES = (p_ws * PS) / ((p_ws * PS) + (p_wh * PH))
    probs.append(p_spam_BAYES)
    print(f'Usando Bayes P(spam|w)={p_spam_BAYES}')
    print(f'Probabilidades de todas las palabras del mail: {probs}')
    clasificacion = multiplica(probs)
    if clasificacion >= 0.5:
        print(f'email es SPAM: P(spam)={clasificacion * 100:.4f}%')
    else:
        print(f'email es HAM: P(spam)={clasificacion * 100:.4f}%')
    return clasificacion

In [28]:
for email in test_spam_stemmed:
    print(f"===== Test mail SPAM {email} =====")
    Bayes(email)

for email in test_ham_stemmed:
    print(f"===== Test mail HAM {email} =====")
    Bayes(email)

===== Test mail SPAM ['password'] =====
P(S)=0.5714285714285714
P(w|spam)=0.16666666666666666
P(H)=0.42857142857142855
P(w|ham)=0.6
Usando Bayes P(spam|w)=0.2702702702702703
Probabilidades de todas las palabras del mail: [0.2702702702702703]
email es HAM: P(spam)=27.0270%
===== Test mail SPAM ['vows'] =====
P(S)=0.5714285714285714
P(w|spam)=0.16666666666666666
P(H)=0.42857142857142855
P(w|ham)=0.6
Usando Bayes P(spam|w)=0.2702702702702703
Probabilidades de todas las palabras del mail: [0.2702702702702703]
email es HAM: P(spam)=27.0270%
===== Test mail HAM ['benefits', 'our', 'account'] =====
P(S)=0.5714285714285714
P(w|spam)=0.16666666666666666
P(H)=0.42857142857142855
P(w|ham)=0.6
Usando Bayes P(spam|w)=0.2702702702702703
Probabilidades de todas las palabras del mail: [0.2702702702702703]
email es HAM: P(spam)=27.0270%
===== Test mail HAM ['importance', 'physical', 'activity'] =====
P(S)=0.5714285714285714
P(w|spam)=0.16666666666666666
P(H)=0.42857142857142855
P(w|ham)=0.6
Usando Baye

## ENTREGA 4: Ejecuta el código y comprueba qué % de emails clasifica bien
Clasifica bien los correos cuya etiqueta real es HAM y mal los correos cuya etiqueta real es SPAM. Teniendo en cuenta que ha tenido 2 aciertos y 2 fallos, clasifica correctamente el 50% de los emails.